<h3 style="color:yellow;font-family:'Arial';">Fuentes</h3>

### Aparcamientos en Madrid
- Madrid - Aparcamientos ocupaciones (rotación): https://datos.madrid.es/portal/site/egob/menuitem.c05c1f754a33a9fbe4b2e4b284f1a5a0/?action=addValoracion&idValorado=44f9b2213c537610VgnVCM1000008a4a900aRCRD&puntuacion=1&vgnextchannel=374512b9ace9f310VgnVCM100000171f5a0aRCRD&vgnextfmt=default&vgnextoid=44f9b2213c537610VgnVCM1000008a4a900aRCRD&utm_source=chatgpt.com

- Zonas del Servicio de Estacionamiento Regulado SER: https://datos.madrid.es/portal/site/egob/menuitem.c05c1f754a33a9fbe4b2e4b284f1a5a0/?vgnextchannel=374512b9ace9f310VgnVCM100000171f5a0aRCRD&vgnextfmt=default&vgnextoid=b9955cde99be2410VgnVCM1000000b205a0aRCRD&utm_source=chatgpt.com

- Servicio de Estacionamiento Regulado (SER). Tiques de aparcamiento: https://datos.madrid.es/portal/site/egob/menuitem.c05c1f754a33a9fbe4b2e4b284f1a5a0/?vgnextchannel=374512b9ace9f310VgnVCM100000171f5a0aRCRD&vgnextfmt=default&vgnextoid=67663c0a55e16710VgnVCM1000001d4a900aRCRD

    > Estructura del Conjunto de Datos:    
     https://datos.madrid.es/FWProjects/egob/Catalogo/Transporte/Ficheros/Estructura_DS_Tiques_Aparcamiento_SER.pdf

### Información metereológica de AEMET
- Acceso a la API:
    > URL: https://opendata.aemet.es/dist/index.html?
    
    > API Key: ---

- Estaciones AEMET:
    > https://www.aemet.es/es/serviciosclimaticos/datosclimatologicos/valoresclimatologicos#tab1

- Datos históricos:
    > valores-climatologicos (/api/valores/climatologicos/diarios/datos/fechaini/{fechaIniStr}/fechafin/{fechaFinStr}/estacion/{idema}): https://opendata.aemet.es/dist/index.html#/valores-climatologicos/Climatolog%C3%ADas%20diarias.

- Predicciones:
    > predicciones-especificas (/api/prediccion/especifica/municipio/horaria/{municipio}): https://opendata.aemet.es/dist/index.html#/predicciones-especificas/Predicci%C3%B3n%20por%20municipios%20horaria.%20Tiempo%20actual.

### Calendario de grandes eventos
- Agenda turística de Madrid (Madrid Convention Bureau):
    > XML: https://www.esmadrid.com/opendata/agenda_v1_es.xml

    > DCAT: https://datos.madrid.es/egob/catalogo/300028-0-agenda-turismo.dcat

    > Estructura del Dataset: https://datos.madrid.es/FWProjects/egob/Catalogo/Turismo/ficheros/Estructura_DS_agenda_turistica.pdf

- IFEMA:
    > Concentra el 70 % de los congresos de más de 5.000 personas en Madrid; su calendario online cubre la mayor parte de los picos de demanda.
    > 

<h3 style="color:yellow;">Librerías</h3>

In [203]:
import pandas as pd
import seaborn as sns
import tqdm
import unicodedata as ud

<div style="color:yellow;">
<h3>
Carga del dataset de tickets del Servicio de Estacionamiento Regulado (SER) de Madrid de 2024
</h3>
Carga de una pequeña <strong>muestra</strong> del dataset completo</strong><br/>
44.000 registros VS aprox 40 millones.
</div>

In [204]:
# Dataset de tiques de parquímetro (SER) de Madrid para 2024
df = pd.read_csv('../data/ser_madrid/2024.csv', parse_dates = ['fecha_operacion', 'fecha_inicio', 'fecha_fin']) # para que sean de tipo datetime
df.head(3)

,matricula_parquimetro,fecha_operacion,fecha_inicio,fecha_fin,cod_distrito,distrito,cod_barrio,barrio,tipo_zona,distintivo,minutos_tique,importe_tique
0,ELPARKING,2024-01-13 12:45:10,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ARGANZUELA,2,ACACIAS,AZUL,ECO,65,"0,30"
1,TELPARK,2024-03-18 14:40:12,2024-03-18 14:40:12,2024-03-18 16:41:12,4,SALAMANCA,2,GOYA,AZUL,C,121,"2,50"
2,EASYPARK,2024-02-27 12:00:12,2024-02-27 12:00:00,2024-02-27 12:20:00,6,TETUAN,4,ALMENARA,AZUL,C,20,"0,60"


In [205]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44000 entries, 0 to 43999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   matricula_parquimetro  44000 non-null  object        
 1   fecha_operacion        44000 non-null  datetime64[ns]
 2   fecha_inicio           44000 non-null  datetime64[ns]
 3   fecha_fin              44000 non-null  datetime64[ns]
 4   cod_distrito           44000 non-null  int64         
 5   distrito               43649 non-null  object        
 6   cod_barrio             44000 non-null  int64         
 7   barrio                 44000 non-null  object        
 8   tipo_zona              44000 non-null  object        
 9   distintivo             44000 non-null  object        
 10  minutos_tique          44000 non-null  int64         
 11  importe_tique          44000 non-null  object        
dtypes: datetime64[ns](3), int64(3), object(6)
memory usage: 4.0+

<h3 style="color:yellow;">Target:</h3>

```%_ocup_barrio(t+1h) = tickets_activos(t+1h) / plazas_totales_barrio```

<h5>En este caso, el número de tickets activos en la próxima hora cómo se calcularía?</h5>

- Un <span style="font-weight:bold;">Ticket Activo</span> es un vehículo que está ocupando una plaza un tiempo determinado.
1. En el dataset tenemos la ````fecha de inicio```` y la ````fecha fin```` de estacionamiento para indicarnos que se trata de un ticket activo.
- Identificar cada ticket con un identificador único.
- **Tickets Activos**: Dataframe con todos los días del año divididos en franjas 15 minutos.
- En ese dataframe de las franjas horarias de 15min, añadimos la información del sumatorio de tickets que están activos durante esa franja, junto a la información relativa al barrio.
- Es decir, no voy a tener en cuenta los tickets que han estado menos de 15 minutos.
> Si un ticket se inica en la hora -1, y su periodo de fin no termina hasta más allá de los 15 minutos de la hora 0, ese ticket estará activo durante esa hora. Tampoco importa si la fecha de finalización del ticket se extiende a la hora +1. A efectos de cálculo, durante esa hora, ese ticket cuanta como ticket activo.

- Al final tendré en una columna con todas las franjas de 15min del año y el número de tickets que han estado activos para cada barrio.

**Cuántos datos debería tener para que el modelo tenga suficientes?**
- El número de plazas disponibles para cada barrio es un dato público y accesible...
- Tendré que utilizar todo el dataset finalmente... a través de Colab no tendré problemas.
- Probablemente sea más complicado la reducción del dataset original sin que se desbalancéen los datos.

> Posible: Clasificación auxiliar: etiqueta binaria alta_ocup (=1 si % ocup ≥ 85 %)... o un semáforo (3 colores)

<h5 style="color:orange;">Identificador único para los tickets</h5>

In [206]:
# añado un identificador único para cada ticket

'''df['ticket_id'] = range(1,len(df)+1)
df.set_index('ticket_id')
df.head(3)'''

"df['ticket_id'] = range(1,len(df)+1)\ndf.set_index('ticket_id')\ndf.head(3)"

<h5 style="color:orange;">Eliminar columnas innecesarias</h5>

In [207]:
# elimino las columnas que no me van a ser útiles.
df.drop(columns=['fecha_operacion', 'cod_distrito', 'distrito', 'importe_tique', 'matricula_parquimetro', 'tipo_zona', 'distintivo'], inplace=True)
df.head(3)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ACACIAS,65
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,GOYA,121
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,ALMENARA,20


<h5 style="color:orange;">
Normalización de columna 'barrio' para poder cruzar datos con el dataset 'plazas_barrios.csv'
</h5>

In [208]:
def normalize_text(text):
    text = text.lower()
    text = ud.normalize(
        'NFKD',
        text).encode('ascii', 'ignore').decode('utf-8')
    return text

# Aplicar la normalización a la columna de nombres de barrio
df['barrio'] = df['barrio'].apply(normalize_text)

# igualo con el dataset de barrios uno de los nombres de barrio manualmente
df['barrio'] = df['barrio'].replace('los carmenes', 'carmenes')

In [209]:
df.head(3)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,almenara,20


In [210]:
df[df['barrio'] == 'carmenes'].head(3)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique
1204,2024-03-23 14:16:00,2024-03-25 09:03:00,1,carmenes,47
1811,2024-03-05 15:37:00,2024-03-05 17:07:00,1,carmenes,90
2768,2024-02-05 10:43:00,2024-02-05 11:18:00,1,carmenes,35


<h5 style="color:orange;">
Tickets Activos si > 15min
</h5>

In [211]:
# Clasifico los tickets como Activos si superan los 15 minutos
df_activos = df[df['minutos_tique'] >= 15].copy()
df_activos.head(3)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,almenara,20


<h5 style="color:orange;">
Slots de 15min
</h5>

In [212]:
# creo una columna 'slot_inicio' para clasificar cada ticket según la 'fecha_inicio'
df_activos['slot_inicio'] = df_activos['fecha_inicio'].dt.floor('15min')
df_activos.head(3)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,slot_inicio
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65,2024-01-13 12:45:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121,2024-03-18 14:30:00
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,almenara,20,2024-02-27 12:00:00


In [213]:
# creo otra columna 'slot_fin' para identificar hasta cuándo está activo cada ticket
df_activos['slot_fin'] = (df_activos['fecha_fin'] - pd.Timedelta(seconds=1)).dt.floor('15min')
df_activos.head()

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,slot_inicio,slot_fin
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65,2024-01-13 12:45:00,2024-01-13 13:45:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121,2024-03-18 14:30:00,2024-03-18 16:30:00
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,almenara,20,2024-02-27 12:00:00,2024-02-27 12:15:00
3,2024-03-18 16:44:00,2024-03-18 17:48:00,4,legazpi,64,2024-03-18 16:30:00,2024-03-18 17:45:00
4,2024-02-01 15:29:00,2024-02-01 15:45:00,3,ciudad universitaria,16,2024-02-01 15:15:00,2024-02-01 15:30:00


In [214]:
# creo otra columna 'slots' que contiene todos los slots en los que está cada ticket
def lista_slots(ticket):
    return pd.date_range(
        start = ticket.slot_inicio,
        end = ticket.slot_fin,
        freq = '15min'
    )

df_activos['slots'] = df_activos.apply(lista_slots, axis=1)
df_activos.head(3)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,slot_inicio,slot_fin,slots
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65,2024-01-13 12:45:00,2024-01-13 13:45:00,"DatetimeIndex(['2024-01-13 12:45:00', '2024-01..."
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121,2024-03-18 14:30:00,2024-03-18 16:30:00,"DatetimeIndex(['2024-03-18 14:30:00', '2024-03..."
2,2024-02-27 12:00:00,2024-02-27 12:20:00,4,almenara,20,2024-02-27 12:00:00,2024-02-27 12:15:00,"DatetimeIndex(['2024-02-27 12:00:00', '2024-02..."


In [215]:
print('Ejemplo de "slots" de un ticket:')
df_activos.slots[0]

Ejemplo de "slots" de un ticket:


DatetimeIndex(['2024-01-13 12:45:00', '2024-01-13 13:00:00',
               '2024-01-13 13:15:00', '2024-01-13 13:30:00',
               '2024-01-13 13:45:00'],
              dtype='datetime64[ns]', freq='15min')

In [216]:
# creo un nuevo dataframe que guarde (explote) una fila por cada slot para cada ticket (para poder agruparlos después)
df_slots = (
    df_activos
        .explode("slots")
        .drop(columns=['slot_inicio'])
        .rename(columns={'slots': 'slot_inicio'})
)
df_slots.head(10)

,fecha_inicio,fecha_fin,cod_barrio,barrio,minutos_tique,slot_fin,slot_inicio
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65,2024-01-13 13:45:00,2024-01-13 12:45:00
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65,2024-01-13 13:45:00,2024-01-13 13:00:00
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65,2024-01-13 13:45:00,2024-01-13 13:15:00
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65,2024-01-13 13:45:00,2024-01-13 13:30:00
0,2024-01-13 12:45:10,2024-01-13 13:50:10,2,acacias,65,2024-01-13 13:45:00,2024-01-13 13:45:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121,2024-03-18 16:30:00,2024-03-18 14:30:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121,2024-03-18 16:30:00,2024-03-18 14:45:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121,2024-03-18 16:30:00,2024-03-18 15:00:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121,2024-03-18 16:30:00,2024-03-18 15:15:00
1,2024-03-18 14:40:12,2024-03-18 16:41:12,2,goya,121,2024-03-18 16:30:00,2024-03-18 15:30:00


In [217]:
############# DATAFRAME BASE DE OCUPACIÓN #############
# agrupo por slots y barrio
'''ocupacion = df_slots.groupby(['cod_barrio', 'slots']).size().reset_index(name='tickets_activos')
df_slots[(df_slots['cod_barrio'] == 11)].head(20)'''

"ocupacion = df_slots.groupby(['cod_barrio', 'slots']).size().reset_index(name='tickets_activos')\ndf_slots[(df_slots['cod_barrio'] == 11)].head(20)"

In [218]:
# 3.1 Agrupar
'''ocup = (df_slotted
        .groupby(["barrio", "slots"])
        .size()
        .reset_index(name="tickets_activos"))'''

'ocup = (df_slotted\n        .groupby(["barrio", "slots"])\n        .size()\n        .reset_index(name="tickets_activos"))'

<div style="color:yellow;">OTRO ACERCAMIENTO:
<h3>
Creo primero el dataframe con todos los slots del año 2024
</h3>
y después relleno.
</div>
<h5 style="color:orange;">
date_range
</h5>

In [219]:
# Guardo todos los slots de 15 min para 2024
rango_datos = pd.date_range(
    start="2024-01-01 00:00",
    end="2024-12-31 23:45",
    freq="15min"
)

rango_datos

DatetimeIndex(['2024-01-01 00:00:00', '2024-01-01 00:15:00',
               '2024-01-01 00:30:00', '2024-01-01 00:45:00',
               '2024-01-01 01:00:00', '2024-01-01 01:15:00',
               '2024-01-01 01:30:00', '2024-01-01 01:45:00',
               '2024-01-01 02:00:00', '2024-01-01 02:15:00',
               ...
               '2024-12-31 21:30:00', '2024-12-31 21:45:00',
               '2024-12-31 22:00:00', '2024-12-31 22:15:00',
               '2024-12-31 22:30:00', '2024-12-31 22:45:00',
               '2024-12-31 23:00:00', '2024-12-31 23:15:00',
               '2024-12-31 23:30:00', '2024-12-31 23:45:00'],
              dtype='datetime64[ns]', length=35136, freq='15min')

<h5 style="color:orange;">
Dataframe base: df_slots
</h5>

In [220]:
# dataframe base con la columna "día de la semana"
'''nombre_dia = {
    0: "lunes",
    1: "martes",
    2: "miércoles",
    3: "jueves",
    4: "viernes",
    5: "sábado",
    6: "domingo"
}'''

calendario = (
    pd.DataFrame(index=rango_datos)
      .assign(dia_semana=rango_datos.day_name("es"))
      .reset_index(names="slot_inicio")      # deja slot_inicio como columna
)

calendario

,slot_inicio,dia_semana
0,2024-01-01 00:00:00,Lunes
1,2024-01-01 00:15:00,Lunes
2,2024-01-01 00:30:00,Lunes
3,2024-01-01 00:45:00,Lunes
4,2024-01-01 01:00:00,Lunes
...,...,...
35131,2024-12-31 22:45:00,Martes
35132,2024-12-31 23:00:00,Martes
35133,2024-12-31 23:15:00,Martes
35134,2024-12-31 23:30:00,Martes


<h5 style="color:orange;">
conteo de todos los slots con tickets activos por barrio
</h5>

In [221]:
conteos = (
    df_slots
        .groupby(['slot_inicio','barrio'])
        .size()
        .reset_index(name='n_tickets_activos')
)

conteos

,slot_inicio,barrio,n_tickets_activos
0,2024-01-02 09:00:00,atocha,1
1,2024-01-02 09:00:00,bellas vistas,1
2,2024-01-02 09:00:00,casa de campo,1
3,2024-01-02 09:00:00,castellana,2
4,2024-01-02 09:00:00,ciudad universitaria,1
...,...,...,...
377720,2025-01-02 09:45:00,hispanoamerica,1
377721,2025-01-02 10:00:00,hispanoamerica,1
377722,2025-01-02 10:15:00,hispanoamerica,1
377723,2025-01-02 10:30:00,hispanoamerica,1


<h5 style="color:orange;">
Dataframe con todos los slots de 2024 rellenos con la información de los tickets activos
</h5>

In [222]:
# 3.1 tabla dinámica: filas = slot, columnas = barrio, valores = nº tickets
# ----------------------------------------------------------------------
# (partimos de `conteos` que tiene las columnas ['slot_inicio', 'barrio', 'n_tickets_activos'])

tabla_tickets = (
    conteos
        .pivot_table(index='slot_inicio',
                      columns='barrio',
                      values='n_tickets_activos',
                      fill_value=0)                # huecos a 0 en el pivoteo
        .reset_index()                             # deja 'slot_inicio' como columna normal
)

# 3.1.1  Asegurarse de que los timestamps son *naive* (sin zona horaria)
#            y por tanto comparables con los de `df_slots`

tabla_tickets['slot_inicio'] = (
    pd.to_datetime(tabla_tickets['slot_inicio'])
      .dt.tz_localize(None)
)

# Unimos calendario de slots + día_semana con los conteos de tickets

tabla = (
    calendario
        .merge(tabla_tickets, on='slot_inicio', how='left')
        .fillna(0)
)

# 3.3 convierte las columnas de barrios a entero
cols_barrios = tabla.columns.difference(['slot_inicio', 'dia_semana'])
tabla[cols_barrios] = tabla[cols_barrios].astype(int)

In [223]:
tabla = tabla.rename(columns={'slot_inicio': 'slot'})

In [224]:
tabla.head(3)

,slot,dia_semana,acacias,adelfas,almagro,almenara,almendrales,arapiles,arguelles,atalaya,...,san isidro,san juan bautista,san pascual,sol,trafalgar,universidad,valdeacederas,valdezarza,vallehermoso,ventas
0,2024-01-01 00:00:00,Lunes,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2024-01-01 00:15:00,Lunes,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2024-01-01 00:30:00,Lunes,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


<h5 style="color:orange;">
Guardo CSV intermedio con los tickets por slots y por barrios
</h5>


In [225]:
#tabla.to_csv('../data/ser_madrid/tickets_slots_barrios.csv', index=False, encoding="utf-8")

<h3 style="color:yellow;">
Dataframe con tickets por horas y barrios (en lugar de slots de 15min)
</h3>

In [226]:
# 1) Índice temporal
tabla = tabla.set_index("slot")

In [227]:
# 2) Mantengo solo columnas numéricas para el resample
num_cols = tabla.select_dtypes("number").columns      # todos los barrios
tickets_1h = tabla[num_cols].resample("H").sum()      # 4 slots = 1 hora

C:\Users\xabi\AppData\Local\Temp\ipykernel_12860\1668799785.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  tickets_1h = tabla[num_cols].resample("H").sum()      # 4 slots = 1 hora


In [228]:
# 3) Añadimos de nuevo la columna 'dia_semana' (en castellano)
tickets_1h["dia_semana"] = tickets_1h.index.day_name("es")

In [229]:
tickets_1h.sample(3)

,acacias,adelfas,almagro,almenara,almendrales,arapiles,arguelles,atalaya,atocha,bellas vistas,...,san juan bautista,san pascual,sol,trafalgar,universidad,valdeacederas,valdezarza,vallehermoso,ventas,dia_semana
slot,,,,,,,,,,,,,,,,,,,,,
2024-08-09 14:00:00,2,1,3,0,0,0,4,0,3,4,...,0,4,0,0,0,0,0,0,0,Viernes
2024-08-27 23:00:00,0,0,4,0,0,0,0,0,0,4,...,0,0,0,0,0,0,0,0,0,Martes
2024-08-14 03:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Miércoles


<h3 style="color:yellow;">
Dataframe con Porcentaje % de ocupación
</h3>

<h5 style="color:orange;">
Cargo el CSV con las plazas por barrio
</h5>

In [230]:
df_plazas_barrios = pd.read_csv('../data/ser_madrid/plazas_barrios.csv')
df_plazas_barrios.sample(3)

,barrio,numero_plazas
32,ibiza,2002
31,hispanoamerica,6079
38,lista,1733


In [231]:
plazas = (df_plazas_barrios.set_index("barrio")["numero_plazas"])

In [232]:
# Cálculo de porcentaje de plazas
ocupacion_1h = tickets_1h.div(plazas, axis='columns') * 100      # cada columna / su capacidad
ocupacion_1h.columns = [f"{c}_porc" for c in ocupacion_1h.columns]  # renombra

In [247]:
ocupacion_1h.head()

,acacias_porc,adelfas_porc,almagro_porc,almenara_porc,almendrales_porc,arapiles_porc,arguelles_porc,atalaya_porc,atocha_porc,bellas vistas_porc,...,san isidro_porc,san juan bautista_porc,san pascual_porc,sol_porc,trafalgar_porc,universidad_porc,valdeacederas_porc,valdezarza_porc,vallehermoso_porc,ventas_porc
slot,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 03:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 04:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [235]:
ocupacion_1h.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 8784 entries, 2024-01-01 00:00:00 to 2024-12-31 23:00:00
Freq: h
Data columns (total 64 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   acacias_porc               8784 non-null   float64
 1   adelfas_porc               8784 non-null   float64
 2   almagro_porc               8784 non-null   float64
 3   almenara_porc              8784 non-null   float64
 4   almendrales_porc           8784 non-null   float64
 5   arapiles_porc              8784 non-null   float64
 6   arguelles_porc             8784 non-null   float64
 7   atalaya_porc               8784 non-null   float64
 8   atocha_porc                8784 non-null   float64
 9   bellas vistas_porc         8784 non-null   float64
 10  berruguete_porc            8784 non-null   float64
 11  carmenes_porc              8784 non-null   float64
 12  casa de campo_porc         8784 non-null   float64
 13  cast

In [233]:
ocupacion_1h.describe()

,acacias_porc,adelfas_porc,almagro_porc,almenara_porc,almendrales_porc,arapiles_porc,arguelles_porc,atalaya_porc,atocha_porc,bellas vistas_porc,...,san isidro_porc,san juan bautista_porc,san pascual_porc,sol_porc,trafalgar_porc,universidad_porc,valdeacederas_porc,valdezarza_porc,vallehermoso_porc,ventas_porc
count,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,...,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000,8784.000000
mean,0.031385,0.025644,0.049019,0.022278,0.000577,0.041353,0.042812,0.014774,0.026359,0.023273,...,0.011384,0.017781,0.015771,0.015299,0.043636,0.014947,0.020511,0.013507,0.041491,0.017410
std,0.059993,0.061970,0.085023,0.047704,0.009099,0.087019,0.086027,0.204236,0.082065,0.060617,...,0.093352,0.051631,0.041174,0.146758,0.085570,0.062211,0.062715,0.053625,0.072326,0.030418
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.028409,0.000000,0.105337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.094787,0.030125
max,0.539773,0.513699,0.667135,0.397163,0.273011,0.693413,0.884956,3.053435,0.760719,0.554236,...,0.865801,0.491803,0.367751,1.581028,0.682439,0.780696,0.739713,0.426569,0.600316,0.241000


In [234]:
# existe algún NaN?
if ocupacion_1h.isna().values.any():
    print("Hay valores faltantes")
    print(ocupacion_1h.isna().sum().sort_values(ascending=False))
else:
    print("Sin NaN: el dataset está completo")


Hay valores faltantes
dia_semana_porc       8784
acacias_porc             0
almagro_porc             0
adelfas_porc             0
almendrales_porc         0
                      ... 
universidad_porc         0
valdeacederas_porc       0
valdezarza_porc          0
vallehermoso_porc        0
ventas_porc              0
Length: 64, dtype: int64


<h3 style="color:yellow;">
Dataframe con un registro por barrio y hora
</h3>

In [238]:
long = (
    ocupacion_1h                  # índice = fecha-hora
      .stack()                    # convierte las columnas en filas (para cada barrio/hora genera una fila)
      .rename("ocupacion_%")      # nueva columna-valor
      .to_frame()                 # convierte la serie en dataframe
      .reset_index()
      .rename(columns={"level_1": "barrio", "slot": "fecha_hora"})
)

long.sample(5)

,fecha_hora,barrio,ocupacion_%
404408,2024-09-24 11:00:00,carmenes_porc,0.0
21181,2024-01-15 00:00:00,castellana_porc,0.676247
121513,2024-03-21 08:00:00,puerta del angel_porc,0.0
407268,2024-09-26 08:00:00,la paz_porc,0.0
212432,2024-05-20 11:00:00,valdeacederas_porc,0.184928


In [239]:
# Añado las variables temporales
long = long.assign(
    hora        = long["fecha_hora"].dt.hour,
    dia_semana  = long["fecha_hora"].dt.dayofweek,
)

In [248]:
# limpiamos _porc de los nombres de los barrios
long["barrio"] = long["barrio"].str.replace(r"_porc$", "", regex=True)

In [249]:
long.sample(3)

,fecha_hora,barrio,ocupacion_%,hora,dia_semana,ocupacion_1h_antes
510776,2024-12-03 19:00:00,justicia,0.0,19,1,0.0
220091,2024-05-25 13:00:00,ibiza,0.1998,13,5,0.0
464117,2024-11-02 22:00:00,valdeacederas,0.0,22,5,0.0


In [245]:
# Lag 1 h por barrio
long["ocupacion_1h_antes"] = (
    long.groupby("barrio")["ocupacion_%"]
        .shift(1)
)
long.head(20)

,fecha_hora,barrio,ocupacion_%,hora,dia_semana,ocupacion_1h_antes
0,2024-01-01,acacias_porc,0.0,0,0,NaN
1,2024-01-01,adelfas_porc,0.0,0,0,NaN
2,2024-01-01,almagro_porc,0.0,0,0,NaN
3,2024-01-01,almenara_porc,0.0,0,0,NaN
4,2024-01-01,almendrales_porc,0.0,0,0,NaN
5,2024-01-01,arapiles_porc,0.0,0,0,NaN
6,2024-01-01,arguelles_porc,0.0,0,0,NaN
7,2024-01-01,atalaya_porc,0.0,0,0,NaN
8,2024-01-01,atocha_porc,0.0,0,0,NaN
9,2024-01-01,bellas vistas_porc,0.0,0,0,NaN


In [252]:
long.query("barrio == 'legazpi' & fecha_hora.between('2024-06-01','2024-06-02')").head(10)


,fecha_hora,barrio,ocupacion_%,hora,dia_semana,ocupacion_1h_antes
229861,2024-06-01 00:00:00,legazpi,0.0,0,5,0.0
229924,2024-06-01 01:00:00,legazpi,0.0,1,5,0.0
229987,2024-06-01 02:00:00,legazpi,0.0,2,5,0.0
230050,2024-06-01 03:00:00,legazpi,0.0,3,5,0.0
230113,2024-06-01 04:00:00,legazpi,0.0,4,5,0.0
230176,2024-06-01 05:00:00,legazpi,0.0,5,5,0.0
230239,2024-06-01 06:00:00,legazpi,0.0,6,5,0.0
230302,2024-06-01 07:00:00,legazpi,0.0,7,5,0.0
230365,2024-06-01 08:00:00,legazpi,0.0,8,5,0.0
230428,2024-06-01 09:00:00,legazpi,0.0,9,5,0.0


In [253]:
#NaN totales por columna
print(long.isna().sum())

# Filas con NaN en cualquiera de las columnas clave
na_rows = long[long[["ocupacion_%", "ocupacion_1h_antes"]].isna().any(axis=1)]
print(na_rows.head())

fecha_hora             0
barrio                 0
ocupacion_%            0
hora                   0
dia_semana             0
ocupacion_1h_antes    63
dtype: int64
  fecha_hora       barrio ocupacion_%  hora  dia_semana ocupacion_1h_antes
0 2024-01-01      acacias         0.0     0           0                NaN
1 2024-01-01      adelfas         0.0     0           0                NaN
2 2024-01-01      almagro         0.0     0           0                NaN
3 2024-01-01     almenara         0.0     0           0                NaN
4 2024-01-01  almendrales         0.0     0           0                NaN


In [254]:
long = long.dropna(subset=["ocupacion_1h_antes"])

In [255]:
long.head()

,fecha_hora,barrio,ocupacion_%,hora,dia_semana,ocupacion_1h_antes
63,2024-01-01 01:00:00,acacias,0.0,1,0,0.0
64,2024-01-01 01:00:00,adelfas,0.0,1,0,0.0
65,2024-01-01 01:00:00,almagro,0.0,1,0,0.0
66,2024-01-01 01:00:00,almenara,0.0,1,0,0.0
67,2024-01-01 01:00:00,almendrales,0.0,1,0,0.0


In [256]:
# Target = ocupación t+1
long["target"] = (
    long.groupby("barrio")["ocupacion_%"]
        .shift(-1)
)

# Limpieza
long = long.dropna(subset=["ocupacion_1h_antes", "target"])


In [259]:
long.info()

<class 'pandas.core.frame.DataFrame'>
Index: 553266 entries, 63 to 553328
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   fecha_hora          553266 non-null  datetime64[ns]
 1   barrio              553266 non-null  object        
 2   ocupacion_%         553266 non-null  object        
 3   hora                553266 non-null  int32         
 4   dia_semana          553266 non-null  int32         
 5   ocupacion_1h_antes  553266 non-null  object        
 6   target              553266 non-null  object        
dtypes: datetime64[ns](1), int32(2), object(4)
memory usage: 29.5+ MB


<h5 style="color:orange;">
Guardo CSV con los tickets por horas y por barrios, con su ocupación porcentual
</h5>


In [260]:

long.to_csv('../data/ser_madrid/tickets_horas_barrios.csv', index=False, encoding='utf-8')